# Public repository note

This notebook is an output-cleared code copy. It requires locally authorised data and is not runnable from the public repository alone. Green Street raw data, intermediate files, derived aggregates and outputs are not distributed.


# 03 Green Street POI Preparation

This notebook prepares the Green Street POI/churn data for later hypothesis testing. It is intentionally a preparation notebook rather than a final results notebook.

Main goals:

1. Understand the current `POI_Churn` extract and separate row-level POI fields from repeated property-level indicators.
2. Build a postcode-to-property mapping from the current Green Street file, as suggested by Green Street.
3. Prepare property-level turnover indicators without double-counting repeated tenant rows.
4. Create a reusable function for the expected historical 2019-2025 POI flat file, so the new data can slot into the workflow when it arrives.
5. Produce clean intermediate tables for later H1/H2/H3 analysis.


In [ ]:
from pathlib import Path
import os
import re
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

warnings.filterwarnings('ignore', category=UserWarning)

BASE = Path(os.environ.get("DISSERTATION_WORKSPACE", Path.cwd().resolve()))
BASE


## 1. File Configuration

The current POI Churn file is used to understand field structure and create a postcode-to-property lookup. The historical 2019-2025 flat file will be added here once Green Street provides it.


In [ ]:
files = {
    'poi_churn_current': 'Greater_London_POI_Churn_2026-07-01-1110.csv',
    'closed_12m': 'Greater_London_POI_Live_2026-07-01-1110(in).csv',
    'greenstreet_dictionary': 'UK-Retail-Analytics-Pro-Data-Dictionary.pdf',
    'lad_boundaries': 'Local_Authority_Districts_May_2024_Boundaries_UK_BGC_3503156029110784919.geojson',
    'office_markets': 'London_Office_Markets_V1.geojson',
    # Set this when the new Green Street historical flat file arrives.
    'poi_historical_2019_2025': None,
}

for key, fname in files.items():
    if fname is None:
        print(f'{key:28s} pending')
    else:
        p = BASE / fname
        print(f'{key:28s} {p.exists()} {p.name}')


## 2. Helper Functions

These functions standardise postcodes, parse dates, create point geometries, and assign Green Street records to LADs and office submarkets.


In [ ]:
def normalize_postcode(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).upper().strip()
    value = re.sub(r'\s+', '', value)
    if value in {'', 'NAN', 'NONE', '<NA>'}:
        return pd.NA
    return value


def parse_date(series):
    # Green Street files mix date styles. infer_datetime_format is deprecated, so use pandas' parser directly.
    return pd.to_datetime(series, errors='coerce')


def first_non_null(series):
    s = series.dropna()
    return s.iloc[0] if len(s) else np.nan


def points_from_lonlat(df, lon='LONGITUDE', lat='LATITUDE', crs='EPSG:4326'):
    clean = df.dropna(subset=[lon, lat]).copy()
    return gpd.GeoDataFrame(clean, geometry=gpd.points_from_xy(clean[lon], clean[lat]), crs=crs)


def load_spatial_layers():
    lad = gpd.read_file(BASE / files['lad_boundaries']).to_crs('EPSG:4326')
    office = gpd.read_file(BASE / files['office_markets']).to_crs('EPSG:4326')

    submarket_crosswalk = {
        'West End': ['Mayfair', 'Soho', "St James's", 'Covent Garden', 'Fitzrovia', 'North of Oxford Street', 'Paddington', 'Knightsbridge', 'Victoria'],
        'City': ['City Core'],
        'Tech Belt & Midtown': ['Midtown', 'Bloomsbury', 'Clerkenwell', 'Euston', 'Kings Cross', 'Shoreditch', 'Camden', 'Aldgate & Whitechapel'],
        'Canary Wharf': ['Canary Wharf'],
        'Southbank': ['Southbank', 'Waterloo', 'Vauxhall, Nine Elms and Battersea'],
    }
    market_to_group = {market: group for group, markets in submarket_crosswalk.items() for market in markets}
    office['study_submarket'] = office['Market'].map(market_to_group).fillna('Outside core / comparison')
    office['inside_core_submarket'] = office['study_submarket'].ne('Outside core / comparison')
    return lad, office


def assign_geographies(points_gdf, lad, office):
    out = gpd.sjoin(
        points_gdf,
        office[['Market', 'study_submarket', 'inside_core_submarket', 'geometry']],
        how='left',
        predicate='intersects',
    ).drop(columns=['index_right'])
    out['study_submarket'] = out['study_submarket'].fillna('Outside office market polygon')
    out['inside_core_submarket'] = out['inside_core_submarket'].fillna(False)
    out = gpd.sjoin(out, lad[['LAD24CD', 'LAD24NM', 'geometry']], how='left', predicate='intersects').drop(columns=['index_right'])
    return out


def weighted_rate(num, den):
    den_sum = den.sum()
    return np.nan if den_sum == 0 else num.sum() / den_sum


## 3. Load Current POI Churn Extract

This file is the latest Green Street POI/churn extract. It should be used to understand the data shape, build a postcode-to-property lookup, and prepare current/recent POI indicators. It should not be treated as a clean historical panel by itself.


In [ ]:
poi = pd.read_csv(BASE / files['poi_churn_current'], low_memory=False)
poi['postcode_norm'] = poi['ZIP'].apply(normalize_postcode)
poi['DATE_CREATE_PARSED'] = parse_date(poi['DATE_CREATE'])
poi['DATE_CLOSE_PARSED'] = parse_date(poi['DATE_CLOSE'])

print('Rows:', len(poi))
print('Columns:', len(poi.columns))
print('Unique tenants:', poi['TENANT_ID'].nunique(dropna=True))
print('Unique premises:', poi['PREMISES_ID'].nunique(dropna=True))
print('Unique properties:', poi['PROPERTY_ID'].nunique(dropna=True))
print('Unique normalised postcodes:', poi['postcode_norm'].nunique(dropna=True))

display(poi['TENANT_STATUS'].value_counts(dropna=False).rename('records').to_frame())
display(poi['PREMISES_STATUS'].value_counts(dropna=False).rename('records').to_frame().head(10))


## 4. Field-Level Audit for the Current Extract

This small table separates likely row-level fields from repeated property-level or geography-level fields. The purpose is to prevent accidental double counting.


In [ ]:
row_level_fields = [
    'AUTO_ID', 'TENANT_ID', 'PREMISES_ID', 'PROPERTY_ID', 'TENANT_STATUS', 'PREMISES_STATUS',
    'CLASSIFICATION', 'CATEGORY', 'SUBCATEGORY', 'SUBCATEGORY_PREVIOUS', 'TENANT',
    'DATE_CREATE', 'DATE_CLOSE', 'DATE_PREMISES_CREATE', 'LATITUDE', 'LONGITUDE', 'ZIP', 'postcode_norm',
    'AREA_SM', 'FLAG_INDEPENDENT', 'FLAG_CONCESSION', 'FLAG_IS_NEW'
]

property_metric_fields = [
    'PROPERTY_ID', 'PROPERTY', 'LATITUDE', 'LONGITUDE', 'postcode_norm',
    'NUMBER_PROPERTY_UNIT', 'NUMBER_PROPERTY_UNIT_CHANGE_OPEN_YOY', 'NUMBER_PROPERTY_UNIT_CHANGE_CLOSE_YOY',
    'NUMBER_PROPERTY_UNIT_CHANGE_YOY', 'NUMBER_PROPERTY_UNIT_GROWTH_OPEN_YOY', 'NUMBER_PROPERTY_UNIT_GROWTH_CLOSE_YOY',
    'NUMBER_PROPERTY_UNIT_GROWTH_YOY', 'RATE_PROPERTY_VACANT', 'RATE_PROPERTY_VACANT_DELTA_YOY',
    'NUMBER_PROPERTY_VACANT_YEAR_1_LT', 'NUMBER_PROPERTY_VACANT_YEAR_2_LT', 'NUMBER_PROPERTY_VACANT_YEAR_3_LT',
    'NUMBER_PROPERTY_VACANT_LT', 'RATE_PROPERTY_VACANT_LT', 'TENANCY_DURATION_DAYS_PROPERTY_AVERAGE',
    'AREA_SM_PROPERTY_AVERAGE', 'SCORE_HEALTH_INDEX', 'SCORE_TAP_SCORE'
]

geography_metric_fields = [
    c for c in poi.columns
    if c.startswith('NUMBER_TENANT') or c.startswith('RATE_VACANT_HIGH_STREET')
    or c.startswith('RATE_VACANT_RETAIL_PARK') or c.startswith('RATE_VACANT_SHOPPING_CENTRE')
    or c == 'RATE_VACANT_GEOGRAPHY_AVERAGE'
]

field_audit = pd.DataFrame({
    'field_group': ['row_level', 'property_metric', 'geography_metric'],
    'n_fields_present': [
        len([c for c in row_level_fields if c in poi.columns]),
        len([c for c in property_metric_fields if c in poi.columns]),
        len([c for c in geography_metric_fields if c in poi.columns]),
    ],
    'example_fields': [
        ', '.join([c for c in row_level_fields if c in poi.columns][:8]),
        ', '.join([c for c in property_metric_fields if c in poi.columns][:8]),
        ', '.join(geography_metric_fields[:8]),
    ]
})
display(field_audit)


## 5. Build Postcode-to-Property Mapping

Green Street suggested using the current file to create a postcode-to-property ID mapping. This is useful for the incoming historical flat file, which may not include `property_id`.

Important rule: a postcode can sometimes map to multiple Green Street properties. Those cases should be flagged as ambiguous rather than forced into a single property ID.


In [ ]:
postcode_property_rows = (
    poi.dropna(subset=['postcode_norm', 'PROPERTY_ID'])
    [['postcode_norm', 'ZIP', 'PROPERTY_ID', 'PROPERTY', 'LATITUDE', 'LONGITUDE']]
    .drop_duplicates()
)

postcode_summary = (
    postcode_property_rows
    .groupby('postcode_norm')
    .agg(
        rows=('PROPERTY_ID', 'size'),
        n_property_ids=('PROPERTY_ID', 'nunique'),
        n_property_names=('PROPERTY', 'nunique'),
    )
    .reset_index()
)

postcode_unique = postcode_property_rows.merge(
    postcode_summary.loc[postcode_summary['n_property_ids'].eq(1), ['postcode_norm']],
    on='postcode_norm',
    how='inner'
).drop_duplicates('postcode_norm')

postcode_ambiguous = postcode_property_rows.merge(
    postcode_summary.loc[postcode_summary['n_property_ids'].gt(1), ['postcode_norm']],
    on='postcode_norm',
    how='inner'
)

print('Postcodes with unique property mapping:', len(postcode_unique))
print('Postcodes with ambiguous property mapping:', postcode_ambiguous['postcode_norm'].nunique())
print('Share ambiguous:', round(postcode_ambiguous['postcode_norm'].nunique() / postcode_summary['postcode_norm'].nunique() * 100, 2), '%')

display(postcode_summary.sort_values('n_property_ids', ascending=False).head(15))


## 6. Current POI Structure by Spatial Unit

This section uses row-level records to describe the current POI structure. For analysis of actual businesses, filter to `TENANT_STATUS == 'Live'`. For current vacancy/demolition structure, keep Live, Vacant, and Demolished but report them separately.


In [ ]:
lad, office = load_spatial_layers()

poi_points = points_from_lonlat(poi)
poi_geo = assign_geographies(poi_points, lad, office)

print('Rows with geometry:', len(poi_geo))
print('Rows with LAD:', poi_geo['LAD24NM'].notna().sum())
print('Rows inside core submarkets:', int(poi_geo['inside_core_submarket'].sum()))

status_by_submarket = pd.crosstab(poi_geo['study_submarket'], poi_geo['TENANT_STATUS'], margins=True).sort_index()
display(status_by_submarket)

live_subcategory = (
    poi_geo[poi_geo['TENANT_STATUS'].eq('Live')]
    .groupby(['study_submarket', 'SUBCATEGORY'])
    .size()
    .rename('live_records')
    .reset_index()
    .sort_values(['study_submarket', 'live_records'], ascending=[True, False])
)
display(live_subcategory.head(30))


## 7. Property-Level Current/Recent Turnover Indicators

The property-level turnover fields are repeated across tenant rows. The code below consolidates them to one record per `PROPERTY_ID` before calculating opening, closure, net change, and turnover rates.


In [ ]:
stable_property_cols = [
    'PROPERTY_ID', 'PROPERTY', 'NUMBER_PROPERTY_UNIT', 'NUMBER_PROPERTY_UNIT_CHANGE_OPEN_YOY',
    'NUMBER_PROPERTY_UNIT_CHANGE_CLOSE_YOY', 'NUMBER_PROPERTY_UNIT_CHANGE_YOY',
    'RATE_PROPERTY_VACANT', 'RATE_PROPERTY_VACANT_LT', 'NUMBER_PROPERTY_VACANT_LT',
    'TENANCY_DURATION_DAYS_PROPERTY_AVERAGE', 'AREA_SM_PROPERTY_AVERAGE', 'SCORE_TAP_SCORE'
]

health_cols = ['SCORE_HEALTH_INDEX']
coord_cols = ['LATITUDE', 'LONGITUDE']

# Check whether candidate property-level fields are stable within PROPERTY_ID.
stability_rows = []
for c in stable_property_cols + health_cols:
    if c in poi.columns:
        nunique_by_property = poi.groupby('PROPERTY_ID')[c].nunique(dropna=True)
        stability_rows.append({
            'field': c,
            'max_distinct_values_within_property': int(nunique_by_property.max()),
            'properties_with_multiple_values': int((nunique_by_property > 1).sum()),
            'share_properties_with_multiple_values': round((nunique_by_property > 1).mean() * 100, 2),
        })
stability_report = pd.DataFrame(stability_rows).sort_values('properties_with_multiple_values', ascending=False)
display(stability_report)


In [ ]:
# Consolidate property-level table. For fields that should be stable, take first non-null.
# For Health Index, keep both first and mean because the current extract can contain multiple values within a property.
agg_spec = {
    'PROPERTY': first_non_null,
    'postcode_norm': first_non_null,
    'LATITUDE': 'median',
    'LONGITUDE': 'median',
    'NUMBER_PROPERTY_UNIT': first_non_null,
    'NUMBER_PROPERTY_UNIT_CHANGE_OPEN_YOY': first_non_null,
    'NUMBER_PROPERTY_UNIT_CHANGE_CLOSE_YOY': first_non_null,
    'NUMBER_PROPERTY_UNIT_CHANGE_YOY': first_non_null,
    'RATE_PROPERTY_VACANT': first_non_null,
    'RATE_PROPERTY_VACANT_LT': first_non_null,
    'NUMBER_PROPERTY_VACANT_LT': first_non_null,
    'TENANCY_DURATION_DAYS_PROPERTY_AVERAGE': first_non_null,
    'AREA_SM_PROPERTY_AVERAGE': first_non_null,
    'SCORE_TAP_SCORE': first_non_null,
    'SCORE_HEALTH_INDEX': ['first', 'mean', 'median'],
}
agg_spec = {k: v for k, v in agg_spec.items() if k in poi.columns}

property_current = poi.groupby('PROPERTY_ID').agg(agg_spec)
property_current.columns = ['_'.join([str(x) for x in col if x]) for col in property_current.columns]
property_current = property_current.reset_index()

# Rename common fields to cleaner names.
property_current = property_current.rename(columns={
    'PROPERTY_first_non_null': 'PROPERTY',
    'postcode_norm_first_non_null': 'postcode_norm',
    'NUMBER_PROPERTY_UNIT_first_non_null': 'number_units',
    'NUMBER_PROPERTY_UNIT_CHANGE_OPEN_YOY_first_non_null': 'opened_units_12m',
    'NUMBER_PROPERTY_UNIT_CHANGE_CLOSE_YOY_first_non_null': 'closed_units_12m',
    'NUMBER_PROPERTY_UNIT_CHANGE_YOY_first_non_null': 'net_unit_change_12m',
    'RATE_PROPERTY_VACANT_first_non_null': 'property_vacancy_rate',
    'RATE_PROPERTY_VACANT_LT_first_non_null': 'property_long_term_vacancy_rate',
    'NUMBER_PROPERTY_VACANT_LT_first_non_null': 'long_term_vacant_units',
    'TENANCY_DURATION_DAYS_PROPERTY_AVERAGE_first_non_null': 'avg_property_tenancy_days',
    'AREA_SM_PROPERTY_AVERAGE_first_non_null': 'avg_property_area_sm',
    'SCORE_TAP_SCORE_first_non_null': 'tap_score',
    'SCORE_HEALTH_INDEX_first': 'health_index_first',
    'SCORE_HEALTH_INDEX_mean': 'health_index_mean',
    'SCORE_HEALTH_INDEX_median': 'health_index_median',
    'LATITUDE_median': 'LATITUDE',
    'LONGITUDE_median': 'LONGITUDE',
})

for c in ['number_units', 'opened_units_12m', 'closed_units_12m', 'net_unit_change_12m']:
    if c in property_current.columns:
        property_current[c] = pd.to_numeric(property_current[c], errors='coerce')

property_current['opening_rate_12m'] = property_current['opened_units_12m'] / property_current['number_units']
property_current['closure_rate_12m'] = property_current['closed_units_12m'] / property_current['number_units']
property_current['net_change_rate_12m'] = property_current['net_unit_change_12m'] / property_current['number_units']
property_current['turnover_rate_12m'] = (property_current['opened_units_12m'] + property_current['closed_units_12m']) / property_current['number_units']

print('Property-level rows:', len(property_current))
display(property_current.head(10))


In [ ]:
property_points = points_from_lonlat(property_current)
property_geo = assign_geographies(property_points, lad, office)

property_turnover_submarket = (
    property_geo
    .groupby('study_submarket')
    .apply(lambda g: pd.Series({
        'properties': g['PROPERTY_ID'].nunique(),
        'units': g['number_units'].sum(),
        'opened_units_12m': g['opened_units_12m'].sum(),
        'closed_units_12m': g['closed_units_12m'].sum(),
        'net_unit_change_12m': g['net_unit_change_12m'].sum(),
        'opening_rate_12m': weighted_rate(g['opened_units_12m'], g['number_units']),
        'closure_rate_12m': weighted_rate(g['closed_units_12m'], g['number_units']),
        'turnover_rate_12m': weighted_rate(g['opened_units_12m'] + g['closed_units_12m'], g['number_units']),
        'mean_property_vacancy_rate': g['property_vacancy_rate'].mean(),
        'mean_long_term_vacancy_rate': g['property_long_term_vacancy_rate'].mean(),
        'mean_health_index': g['health_index_mean'].mean(),
    }))
    .reset_index()
    .sort_values('turnover_rate_12m', ascending=False)
)

display(property_turnover_submarket)


## 8. Historical POI Flat File Template

When Green Street provides the 2019-2025 historical flat file, set the filename in `files['poi_historical_2019_2025']` above and run this section. The expected fields are:

- premises_id
- tenant_id
- tenant_status
- premises_status
- category
- subcategory
- latitude/longitude or postcode
- date_create
- date_close where available

The function below normalises fields, maps postcodes to property IDs where possible, assigns geographies, and constructs opening/closure event tables.


In [ ]:
def standardise_historical_columns(df):
    # Convert common incoming lower-case names to the all-caps convention used in the current POI file.
    rename = {c: c.upper() for c in df.columns}
    df = df.rename(columns=rename)
    aliases = {
        'POSTCODE': 'ZIP',
        'LAT': 'LATITUDE',
        'LON': 'LONGITUDE',
        'LONG': 'LONGITUDE',
    }
    df = df.rename(columns={k: v for k, v in aliases.items() if k in df.columns and v not in df.columns})
    return df


def prepare_historical_poi(path, postcode_unique_lookup, postcode_ambiguous_lookup, lad, office):
    hist = pd.read_csv(path, low_memory=False)
    hist = standardise_historical_columns(hist)

    if 'ZIP' in hist.columns:
        hist['postcode_norm'] = hist['ZIP'].apply(normalize_postcode)
    else:
        hist['postcode_norm'] = pd.NA

    for c in ['DATE_CREATE', 'DATE_CLOSE']:
        if c in hist.columns:
            hist[f'{c}_PARSED'] = parse_date(hist[c])
        else:
            hist[f'{c}_PARSED'] = pd.NaT

    # Map property_id from current postcode lookup if the historical file does not include it.
    if 'PROPERTY_ID' not in hist.columns:
        hist = hist.merge(
            postcode_unique_lookup[['postcode_norm', 'PROPERTY_ID', 'PROPERTY']],
            on='postcode_norm',
            how='left',
        )
        ambiguous_postcodes = set(postcode_ambiguous_lookup['postcode_norm'].dropna().unique())
        hist['property_mapping_status'] = np.select(
            [hist['PROPERTY_ID'].notna(), hist['postcode_norm'].isin(ambiguous_postcodes), hist['postcode_norm'].isna()],
            ['matched_unique_postcode', 'ambiguous_postcode', 'missing_postcode'],
            default='unmatched_postcode_or_new_property'
        )
    else:
        hist['property_mapping_status'] = 'property_id_supplied'

    # Spatial assignment can use coordinates even if property_id mapping is incomplete.
    if {'LATITUDE', 'LONGITUDE'}.issubset(hist.columns):
        hist_points = points_from_lonlat(hist)
        hist_geo = assign_geographies(hist_points, lad, office)
    else:
        hist_geo = hist.copy()
        hist_geo['study_submarket'] = pd.NA
        hist_geo['inside_core_submarket'] = pd.NA
        hist_geo['LAD24CD'] = pd.NA
        hist_geo['LAD24NM'] = pd.NA

    return hist_geo


def build_open_close_events(hist_geo, start_year=2019, end_year=2025):
    events = []
    id_cols = [c for c in ['PREMISES_ID', 'TENANT_ID', 'PROPERTY_ID', 'LAD24NM', 'study_submarket', 'CATEGORY', 'SUBCATEGORY', 'postcode_norm', 'property_mapping_status'] if c in hist_geo.columns]

    if 'DATE_CREATE_PARSED' in hist_geo.columns:
        openings = hist_geo.loc[hist_geo['DATE_CREATE_PARSED'].notna(), id_cols + ['DATE_CREATE_PARSED']].copy()
        openings['event_date'] = openings['DATE_CREATE_PARSED']
        openings['event_type'] = 'opening'
        openings = openings.drop(columns=['DATE_CREATE_PARSED'])
        events.append(openings)

    if 'DATE_CLOSE_PARSED' in hist_geo.columns:
        closures = hist_geo.loc[hist_geo['DATE_CLOSE_PARSED'].notna(), id_cols + ['DATE_CLOSE_PARSED']].copy()
        closures['event_date'] = closures['DATE_CLOSE_PARSED']
        closures['event_type'] = 'closure'
        closures = closures.drop(columns=['DATE_CLOSE_PARSED'])
        events.append(closures)

    if not events:
        return pd.DataFrame()

    events = pd.concat(events, ignore_index=True)
    events['event_year'] = pd.to_datetime(events['event_date'], errors='coerce').dt.year
    events = events[events['event_year'].between(start_year, end_year)].copy()
    return events


In [ ]:
historical_file = files.get('poi_historical_2019_2025')

if historical_file is None:
    print('Historical POI flat file is not available yet. Set files[\'poi_historical_2019_2025\'] when it arrives.')
else:
    hist_geo = prepare_historical_poi(BASE / historical_file, postcode_unique, postcode_ambiguous, lad, office)
    events = build_open_close_events(hist_geo)

    print('Historical records:', len(hist_geo))
    print('Events in analysis period:', len(events))
    display(hist_geo['property_mapping_status'].value_counts(dropna=False).rename('records').to_frame())

    annual_churn = (
        events
        .groupby(['event_year', 'study_submarket', 'event_type'])
        .size()
        .rename('events')
        .reset_index()
    )
    display(annual_churn.head(30))

    subcategory_churn = (
        events
        .groupby(['event_year', 'event_type', 'SUBCATEGORY'])
        .size()
        .rename('events')
        .reset_index()
        .sort_values(['event_year', 'event_type', 'events'], ascending=[True, True, False])
    )
    display(subcategory_churn.head(50))


## 9. Outputs for Later Methodology and Hypothesis Testing

Once the historical flat file is available, the outputs from this notebook should feed into the formal indicator notebook.

Planned outputs:

- `postcode_unique`: postcode-to-property lookup for clean one-to-one matches.
- `postcode_ambiguous`: postcode-to-property cases requiring caution or spatial/manual handling.
- `poi_geo`: row-level current POI records assigned to LAD/submarket.
- `property_geo`: property-level current/recent turnover indicators assigned to LAD/submarket.
- `hist_geo`: historical POI records assigned to LAD/submarket after the new flat file arrives.
- `events`: annual opening and closure events derived from `date_create` and `date_close`.

Role in the dissertation:

- H1: openings/closures and category change as retail adaptation indicators.
- H2: property-level turnover and health as retail resilience outcomes, alongside OpenLocal office restructuring.
- H3: annual churn and pathway classification if the historical flat file is sufficiently complete.


In [ ]:
method_use_table = pd.DataFrame([
    {
        'output': 'postcode_unique / postcode_ambiguous',
        'use': 'Map historical POI records to Green Street property IDs where possible.',
        'caution': 'Ambiguous and unmatched postcodes should be flagged, not silently forced.'
    },
    {
        'output': 'poi_geo',
        'use': 'Describe current POI structure by LAD and office submarket.',
        'caution': 'Filter TENANT_STATUS == Live when analysing actual businesses.'
    },
    {
        'output': 'property_geo',
        'use': 'Measure 2026 endpoint/recent turnover, vacancy, long-term vacancy and Health Index by property/submarket.',
        'caution': 'Property-level fields must be de-duplicated by PROPERTY_ID before aggregation.'
    },
    {
        'output': 'events',
        'use': 'Construct annual openings, closures, net change and churn by category/subcategory after historical file arrives.',
        'caution': 'Interpret date_create as first recorded occupier date unless Green Street confirms it is a true opening date.'
    },
])
display(method_use_table)


## 10. Save Prepared Local Outputs

These outputs are restricted intermediate files for local dissertation analysis. They should not be uploaded to a public GitHub repository because they contain Green Street property identifiers, postcodes, and/or commercial data-derived indicators.


In [ ]:
OUTPUT_DIR = BASE / 'outputs' / 'restricted_greenstreet_prepared'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

postcode_unique_path = OUTPUT_DIR / 'greenstreet_postcode_property_mapping_unique.csv'
postcode_ambiguous_path = OUTPUT_DIR / 'greenstreet_postcode_property_mapping_ambiguous.csv'
postcode_summary_path = OUTPUT_DIR / 'greenstreet_postcode_property_mapping_summary.csv'
property_current_path = OUTPUT_DIR / 'greenstreet_property_current_2026_turnover_indicators.csv'
property_geo_path = OUTPUT_DIR / 'greenstreet_property_current_2026_turnover_indicators_with_geography.csv'
poi_status_submarket_path = OUTPUT_DIR / 'greenstreet_current_poi_tenant_status_by_submarket.csv'

postcode_unique.to_csv(postcode_unique_path, index=False)
postcode_ambiguous.to_csv(postcode_ambiguous_path, index=False)
postcode_summary.to_csv(postcode_summary_path, index=False)
property_current.to_csv(property_current_path, index=False)

property_geo_export = property_geo.drop(columns='geometry', errors='ignore')
property_geo_export.to_csv(property_geo_path, index=False)
status_by_submarket.to_csv(poi_status_submarket_path)

output_manifest = pd.DataFrame([
    {
        'file': postcode_unique_path.name,
        'description': 'One-to-one postcode-to-Green-Street-property mapping from current POI extract.',
        'use': 'Use to lookup PROPERTY_ID for historical POI records where the incoming flat file only has postcode.'
    },
    {
        'file': postcode_ambiguous_path.name,
        'description': 'Postcodes linked to multiple Green Street properties.',
        'use': 'Flag rather than force-match; may require coordinates, manual review, or exclusion in sensitivity checks.'
    },
    {
        'file': postcode_summary_path.name,
        'description': 'Mapping quality summary by postcode.',
        'use': 'Diagnose how many properties each postcode maps to.'
    },
    {
        'file': property_current_path.name,
        'description': 'De-duplicated property-level current/recent Green Street indicators.',
        'use': 'Use for endpoint checks; do not treat as a 2019-2025 historical churn panel.'
    },
    {
        'file': property_geo_path.name,
        'description': 'Property-level current/recent indicators with LAD and office submarket assignment.',
        'use': 'Use for coverage and current structure checks by geography.'
    },
    {
        'file': poi_status_submarket_path.name,
        'description': 'Current POI tenant-status composition by office submarket.',
        'use': 'Use for Chapter 3 data composition figure/table if needed.'
    },
])

manifest_path = OUTPUT_DIR / 'manifest.csv'
output_manifest.to_csv(manifest_path, index=False)

print('Saved restricted local outputs to:', OUTPUT_DIR)
display(output_manifest)
